> Os exemplos desta página usam dados sintéticos gerados pelo projeto. Não são observações ASTER nem permitem interpretar valores como reflectância, temperatura ou classes reais. Veja [os dados de exemplo](../../../../data/README.md).



# Capítulo 03 — Transformações espectrais (PCA)

> **Referência:** Canty (2019), Cap. 3.  
> Teoria: `../aulas/pca.md` (PCA), `../aulas/section_3.5.md` (MNF)

Neste notebook implementamos **PCA do zero** no raster sintético, visualizamos PC1–PC3 e o gráfico de scree.

## 1. Intuição

A PCA encontra um novo sistema de eixos ortogonais no espaço das bandas, ordenados por variância. As primeiras componentes concentram contraste útil; as últimas, em geral, ruído.

A **MNF** (seção 3.5) ordena por razão sinal/ruído e exige estimar $\mathbf{S}_N$ — ver `ex03_2.py`.

## 2. Matemática (resumo)

1. Centre as bandas.  
2. $\mathbf{S}=\mathrm{Cov}(\mathbf{g})$.  
3. $\mathbf{S}\mathbf{w}=\lambda\mathbf{w}$, ordene $\lambda$ decrescente.  
4. $\mathbf{y}=\mathbf{W}^\top(\mathbf{g}-\bar{\mathbf{g}})$.

Detalhes em `../aulas/pca.md`.

## 3. Implementação — PCA no raster sintético

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_utils").is_dir())
DATA = REPO / "data/exemplos/exemplo_sintetico.tif"
OUT = REPO / "conteudo/analise-de-imagens/transformacoes/figuras"
OUT.mkdir(parents=True, exist_ok=True)


def load_raster(path):
    try:
        import rasterio

        with rasterio.open(path) as src:
            return src.read().astype(np.float64)
    except Exception as e_rio:
        rio_err = e_rio
    try:
        from osgeo import gdal
        from osgeo.gdalconst import GA_ReadOnly

        gdal.AllRegister()
        ds = gdal.Open(str(path), GA_ReadOnly)
        if ds is None:
            raise RuntimeError(f"GDAL None: {path}")
        bands, rows, cols = ds.RasterCount, ds.RasterYSize, ds.RasterXSize
        data = np.empty((bands, rows, cols), dtype=np.float64)
        for b in range(bands):
            data[b] = ds.GetRasterBand(b + 1).ReadAsArray().astype(np.float64)
        ds = None
        return data
    except Exception as e_gdal:
        raise RuntimeError(
            f"Instale rasterio ou GDAL. rasterio={rio_err}; gdal={e_gdal}"
        )


def stretch01(band):
    mn, mx = np.nanmin(band), np.nanmax(band)
    if mx <= mn:
        return np.zeros_like(band)
    return (band - mn) / (mx - mn)


image = load_raster(DATA)
nbands, nrows, ncols = image.shape
print(f"{DATA.name}: {image.shape}")

G = image.reshape(nbands, -1).copy()
G -= G.mean(axis=1, keepdims=True)
S = np.cov(G, ddof=1)
lamda, W = np.linalg.eigh(S)
idx = np.argsort(lamda)[::-1]
lamda = lamda[idx]
W = W[:, idx]

n_show = min(3, nbands)
pcs = (W[:, :n_show].T @ G).reshape(n_show, nrows, ncols)
var_explained = lamda / lamda.sum()
print("Autovalores:", np.round(lamda, 4))
print("% variância:", np.round(100 * var_explained, 2))

## 4. Visualizações — PC1–PC3 e scree

In [ ]:
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    axes[i].imshow(stretch01(pcs[i]), cmap="gray")
    axes[i].set_title(f"PC{i+1} (λ={lamda[i]:.2f})")
    axes[i].set_axis_off()
fig.suptitle("Exemplo sintético — primeiras componentes principais", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "pcs_top3.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print(f"Salvo {OUT / 'pcs_top3.png'}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, nbands + 1), lamda, "o-", color="steelblue")
ax.set_xlabel("Componente")
ax.set_ylabel("Autovalor λ")
ax.set_title("Scree plot — PCA Exemplo sintético")
ax.set_xticks(np.arange(1, nbands + 1))
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "pca_scree.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print(f"Salvo {OUT / 'pca_scree.png'}")

# também PNGs individuais (úteis para o mestrado / relatório)
for i in range(n_show):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(stretch01(pcs[i]), cmap="gray")
    ax.set_title(f"PC{i+1}")
    ax.set_axis_off()
    fig.savefig(OUT / f"pc{i+1}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

## 5. Exercícios

```bash
cd ../exercicios

# Erro de reconstrução PCA (r PCs) vs fórmula com autovalores
python ex03_1.py
python ex03_1.py ../../../../data/exemplos/exemplo_sintetico.tif -r 3

# Covariância de ruído (insumo MNF)
python ex03_2.py

# Exporta PC1–PC3 (PNG + GeoTIFF se possível)
python ex03_3.py
```